# Travel Analytics Capstone — Project Setup
**Project summary:** This notebook develops one component of a travel analytics system using the supplied CSV datasets. The broader project combines flight-price regression, gender classification, hotel recommendations, a Flask REST API, a Streamlit dashboard, MLflow tracking, Docker, Kubernetes, Airflow and CI/CD.

**GitHub repository:** `https://github.com/YOUR-USERNAME/travel-analytics-capstone`

> Before submission, replace the placeholder above with the one repository URL containing the full project. Keep all three notebooks pointed to the same repository.

In [ ]:
# In Google Colab, upload the CSVs in the Files panel or mount Google Drive.
# If the repository ZIP is uploaded to Colab, unzip it and set DATA_DIR accordingly.
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path("/content/data")  # Change to the folder containing flights.csv, hotels.csv, users.csv
if not (DATA_DIR / "flights.csv").exists():
    # Local notebook fallback: place CSVs in ./data
    DATA_DIR = Path("data")

flights = pd.read_csv(DATA_DIR / "flights.csv")
hotels = pd.read_csv(DATA_DIR / "hotels.csv")
users = pd.read_csv(DATA_DIR / "users.csv")

print("Flights:", flights.shape)
print("Hotels:", hotels.shape)
print("Users:", users.shape)
display(flights.head())
display(hotels.head())
display(users.head())

## Data understanding and quality checks

The flight dataset contains route, flight type, duration, distance, agency, date and observed price. The hotel dataset contains hotel, place, stay duration, daily price and total price. The users dataset contains age, company and a recorded gender label.

The first checks validate schema, duplicates, missing values, numeric types and date parsing. These checks matter because model quality depends on consistent input definitions; a pipeline that silently changes column types can fail after deployment.

In [ ]:
for name, frame in {"flights": flights, "hotels": hotels, "users": users}.items():
    print(f"\\n{name}: {frame.shape}")
    display(frame.dtypes.rename("dtype").to_frame())
    print("Missing values:", int(frame.isna().sum().sum()))
    print("Duplicate rows:", int(frame.duplicated().sum()))

display(flights.describe(include="all").T)

# 3. Recommender — Hotel Suggestions

## Objective and reasoning
The recommender uses the observed hotel booking history. It aggregates bookings by hotel and location, then balances booking popularity against average daily price. If a user code is supplied, hotels and destinations seen in that user's historical records receive a preference boost. Optional place and maximum daily-price filters help translate the model into an actionable shortlist.

This is a transparent, lightweight ranking baseline. It is not a neural recommender and does not establish that a hotel is currently available or that a historical price is current. The dataset does not include explicit ratings, so booking frequency is used as an implicit preference signal.

In [ ]:
hotel_df = hotels.copy()
for c in ["price", "total", "days"]:
    hotel_df[c] = pd.to_numeric(hotel_df[c], errors="coerce")
hotel_df = hotel_df.dropna(subset=["name", "place", "price"])
hotel_stats = hotel_df.groupby(["name", "place"], as_index=False).agg(
    average_price_per_day=("price", "mean"),
    average_days=("days", "mean"),
    average_total=("total", "mean"),
    bookings=("travelCode", "count")
)
hotel_stats["popularity_percentile"] = hotel_stats["bookings"].rank(pct=True)
hotel_stats["price_percentile"] = hotel_stats["average_price_per_day"].rank(pct=True)
hotel_stats["score"] = 0.65 * hotel_stats["popularity_percentile"] + 0.35 * (1 - hotel_stats["price_percentile"])
display(hotel_stats.sort_values("score", ascending=False).head(10))

## Example recommendation function

The following function illustrates the ranking logic used by the Flask API. User-history boosts should be evaluated on a temporal holdout: hide a user's later booking, generate recommendations from earlier bookings, and measure whether the held-out hotel appears near the top.

In [ ]:
def recommend(hotels_frame, user_code=None, place=None, max_price=None, limit=5):
    d = hotels_frame.copy()
    d["price"] = pd.to_numeric(d["price"], errors="coerce")
    d = d.dropna(subset=["name", "place", "price"])
    if place:
        d = d[d["place"].astype(str).str.contains(str(place), case=False, na=False)]
    if max_price is not None:
        d = d[d["price"] <= float(max_price)]
    if d.empty:
        return pd.DataFrame()
    agg = d.groupby(["name", "place"], as_index=False).agg(
        average_price_per_day=("price", "mean"),
        average_days=("days", "mean"),
        average_total=("total", "mean"),
        bookings=("travelCode", "count")
    )
    agg["score"] = 0.65 * agg["bookings"].rank(pct=True) + 0.35 * (1 - agg["average_price_per_day"].rank(pct=True))
    if user_code is not None:
        history = hotels_frame[pd.to_numeric(hotels_frame["userCode"], errors="coerce") == int(user_code)]
        agg.loc[agg["place"].isin(history["place"].dropna()), "score"] += 0.25
        agg.loc[agg["name"].isin(history["name"].dropna()), "score"] += 0.35
    return agg.sort_values(["score", "bookings"], ascending=False).head(limit)

display(recommend(hotels, user_code=0, limit=5))

## Recommendation evaluation plan

A recommendation model should be evaluated by ranking quality, not only by inspecting examples. Use a time-aware holdout for each user where possible. Report Precision@K and Recall@K, catalog coverage, and the share of recommendations that are already known to the user. Avoid randomly splitting booking rows when that would allow future bookings to influence the recommendation ranking.

In [ ]:
# Simple dataset insights for the Streamlit dashboard
top_places = hotels["place"].value_counts().head(10)
top_places.plot(kind="bar", figsize=(10, 4), title="Most frequently booked locations")
plt.ylabel("Booking rows")
plt.tight_layout()
plt.show()

daily_price = hotel_df.groupby("place")["price"].mean().sort_values().head(15)
display(daily_price.rename("mean_daily_price").to_frame())

## Integration handoff
The shared repository contains `src/travel_analytics/recommend.py`, which is called by the Flask `/recommendations` endpoint and the Streamlit UI. The same repository also contains model training, the API, Docker, Kubernetes manifests, the Airflow DAG, Jenkins pipeline and MLflow logging. Replace the repository placeholder in the notebook introduction with the actual single GitHub repository URL before submission.